In [2]:
import pandas as pd

In [ ]:
path = "../data/en.openfoodfacts.org.products.csv/en.openfoodfacts.org.products.csv"
df = pd.read_csv(path, nrows=100, sep='\t', encoding="utf-8")

: 

: 

: 

: 

In [ ]:
df

,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,last_modified_by,last_updated_t,last_updated_datetime,...,water-hardness_100g,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g,carbohydrates-total_100g
0,2,http://world-en.openfoodfacts.org/product/0000...,openfoodfacts-contributors,1760861583,2025-10-19T08:13:03Z,1760861586,2025-10-19T08:13:06Z,NaN,1760861586,2025-10-19T08:13:06Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,http://world-en.openfoodfacts.org/product/0000...,openfoodfacts-contributors,1752485388,2025-07-14T09:29:48Z,1752485389,2025-07-14T09:29:49Z,NaN,1752485389,2025-07-14T09:29:49Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4,http://world-en.openfoodfacts.org/product/0000...,openfoodfacts-contributors,1768903196,2026-01-20T09:59:56Z,1768903204,2026-01-20T10:00:04Z,NaN,1768903204,2026-01-20T10:00:04Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,http://world-en.openfoodfacts.org/product/0000...,moon-rabbit,1767072228,2025-12-30T05:23:48Z,1767072233,2025-12-30T05:23:53Z,moon-rabbit,1767072233,2025-12-30T05:23:53Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,http://world-en.openfoodfacts.org/product/0000...,moon-rabbit,1760212975,2025-10-11T20:02:55Z,1760218930,2025-10-11T21:42:10Z,ascharao,1760218930,2025-10-11T21:42:10Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,98,http://world-en.openfoodfacts.org/product/0000...,ptitof35,1559383944,2019-06-01T10:12:24Z,1727965061,2024-10-03T14:17:41Z,fix-code-bot,1743160951,2025-03-28T11:22:31Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,99,http://world-en.openfoodfacts.org/product/0000...,elcoco,1566888090,2019-08-27T06:41:30Z,1730038053,2024-10-27T14:07:33Z,smoothie-app,1743690944,2025-04-03T14:35:44Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,100,http://world-en.openfoodfacts.org/product/0000...,del51,1444572561,2015-10-11T14:09:21Z,1737486361,2025-01-21T19:06:01Z,NaN,1743522369,2025-04-01T15:46:09Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,101,http://world-en.openfoodfacts.org/product/0000...,openfoodfacts-contributors,1568746303,2019-09-17T18:51:43Z,1736848791,2025-01-14T09:59:51Z,prepperapp,1743319994,2025-03-30T07:33:14Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


: 

: 

: 

: 

In [ ]:
df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 209 columns):
 #    Column                            Dtype  
---   ------                            -----  
 0    code                              int64  
 1    url                               object 
 2    creator                           object 
 3    created_t                         int64  
 4    created_datetime                  object 
 5    last_modified_t                   int64  
 6    last_modified_datetime            object 
 7    last_modified_by                  object 
 8    last_updated_t                    int64  
 9    last_updated_datetime             object 
 10   product_name                      object 
 11   abbreviated_product_name          float64
 12   generic_name                      object 
 13   quantity                          object 
 14   packaging                         object 
 15   packaging_tags                    object 
 16   packaging_en             

: 

: 

: 

: 

In [ ]:
# Variables les plus corrélées au Nutri-Score et à l'Eco-Score
import numpy as np

# Candidats de colonnes cibles (OpenFoodFacts peut varier selon version)
nutri_candidates = ["nutriscore_score", "nutrition-score-fr_100g", "nutriscore"]
eco_candidates = ["ecoscore_score", "ecoscore_data.grade", "ecoscore"]

def pick_existing_column(candidates, columns):
    for c in candidates:
        if c in columns:
            return c
    return None

nutri_col = pick_existing_column(nutri_candidates, df.columns)
eco_col = pick_existing_column(eco_candidates, df.columns)

def show_strongest_correlations(target_col, data, min_periods=5):
    if target_col is None:
        print("Colonne cible introuvable.")
        return

    # Garder uniquement les variables numériques
    num_df = data.select_dtypes(include=[np.number]).copy()

    # Au cas où la cible est en texte mais convertible en nombre
    if target_col not in num_df.columns and target_col in data.columns:
        converted = pd.to_numeric(data[target_col], errors="coerce")
        num_df[target_col] = converted

    if target_col not in num_df.columns:
        print(f"La colonne '{target_col}' n'est pas numérique et n'a pas pu être convertie.")
        return

    corr_matrix = num_df.corr(numeric_only=True, min_periods=min_periods)
    if target_col not in corr_matrix.columns:
        print("Impossible de calculer les corrélations pour cette cible (données insuffisantes).")
        return

    corr = corr_matrix[target_col].dropna().drop(labels=[target_col], errors="ignore")
    if corr.empty:
        print("Aucune variable corrélée calculable avec les paramètres actuels.")
        return

    result = pd.DataFrame({
        "correlation": corr,
        "correlation_abs": corr.abs()
    }).sort_values(["correlation_abs", "correlation"], ascending=[False, False])

    max_corr = result["correlation_abs"].max()
    strongest = result[np.isclose(result["correlation_abs"], max_corr)]

    print(f"\nVariable(s) LES PLUS corrélée(s) à '{target_col}' (|corr| max = {max_corr:.4f}) :")
    display(strongest)

    print("\nClassement complet (trié par corrélation absolue décroissante) :")
    display(result)

print(f"Colonne Nutri-Score utilisée : {nutri_col}")
show_strongest_correlations(nutri_col, df)

print(f"\nColonne Eco-Score utilisée : {eco_col}")
show_strongest_correlations(eco_col, df)

Colonne Nutri-Score utilisée : nutriscore_score

Top 15 variables corrélées à 'nutriscore_score' :


,correlation,correlation_abs
environmental_score_score,-0.478611,0.478611
last_image_t,0.282933,0.282933
last_updated_t,-0.250625,0.250625
product_quantity,0.203863,0.203863
completeness,0.155171,0.155171
code,0.146941,0.146941
nova_group,0.128811,0.128811
serving_quantity,-0.100111,0.100111
additives_n,0.089867,0.089867
unique_scans_n,0.066886,0.066886



Colonne Eco-Score utilisée : None
Colonne cible introuvable.


: 

: 

: 

: 